# Workshop 1 · Kimball Model Build

## Workshop Scenario

RetailHub wants one reusable Gold source for Power BI instead of every report
joining Silver tables independently. Your task is to build the first BI-ready
star schema: dimensions, an order-line fact table and a denormalized table for
fast report prototyping.


## Success Criteria

You are done when:

1. five Gold objects exist: `dim_date`, `dim_customer`, `dim_product`,
   `fact_sales`, `fact_sales_dashboard`,
2. `fact_sales` and `fact_sales_dashboard` are both one row per `line_id`,
3. completed revenue reconciles between fact and dashboard table,
4. dimension attributes needed by Power BI are present,
5. the final self-check cell passes without exceptions.


## Target Gold Contract

| Object | Grain | BI purpose |
|---|---|---|
| `gold.dim_date` | one row per order date | calendar hierarchy and month filters |
| `gold.dim_customer` | one row per customer | segment and region slicers |
| `gold.dim_product` | one row per product | category and product slicers |
| `gold.fact_sales` | one row per order line | governed measures and relationships |
| `gold.fact_sales_dashboard` | one row per order line, denormalized | quick Power BI connection and validation |

Do not optimize for clever SQL. Optimize for a model that another BI developer
can understand in five minutes.


**Goal:** Build a complete star schema in your Gold layer — 5 objects that Power BI will consume. Workshop 2 (Day 2) adds the KPI layer on top.

In [ ]:
%run ../../setup/00_setup


### Environment config

Verify that the setup resolved your personal catalog and all three schemas.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("SILVER",  SILVER),
    ("GOLD",    GOLD),
], ["Variable", "Value"]))


### Silver pre-check

All four Silver source tables must exist before we build Gold. Fail fast if any are missing.

In [ ]:
silver_tables = [f"{SILVER}.customers", f"{SILVER}.products",
                 f"{SILVER}.sales_orders", f"{SILVER}.order_lines"]
missing = [t for t in silver_tables if not spark.catalog.tableExists(t)]
if missing:
    for t in missing: print("[MISSING]", t)
    raise Exception("Silver pre-check failed. Run data generation first.")
print("[OK] All Silver source tables exist.")


## Task 1 — dim_date

Source: distinct order dates from silver.sales_orders. We extract year, quarter, month, year_month string, and day_of_week for BI date hierarchy.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_date
COMMENT 'Date dimension for RetailHub Power BI reports.' AS
SELECT DISTINCT
  order_date                            AS date_key,
  YEAR(order_date)                      AS year,
  QUARTER(order_date)                   AS quarter,
  MONTH(order_date)                     AS month,
  DATE_FORMAT(order_date, 'yyyy-MM')    AS year_month,
  DAYOFWEEK(order_date)                 AS day_of_week
FROM silver.sales_orders
WHERE order_date IS NOT NULL
ORDER BY date_key


Verify row count and date range.

In [ ]:
%sql
SELECT COUNT(*) AS rows, MIN(date_key) AS min_date, MAX(date_key) AS max_date
FROM gold.dim_date


Inspect the most recent dates to confirm day_of_week and year_month formatting.

In [ ]:
%sql
SELECT * FROM gold.dim_date ORDER BY date_key DESC LIMIT 5


## Task 2 — dim_customer

Source: silver.customers. One row per customer. region becomes customer_region for clarity in BI.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_customer
COMMENT 'Customer dimension with BI-friendly region naming.' AS
SELECT
  customer_id,
  customer_name,
  segment,
  region AS customer_region
FROM silver.customers


Verify row count and uniqueness of customer_id.

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS unique_customers
FROM gold.dim_customer


Preview dimension rows.

In [ ]:
%sql
SELECT * FROM gold.dim_customer LIMIT 5


## Task 3 — dim_product

Source: silver.products. unit_cost is carried forward — needed to compute line_margin in fact_sales.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_product
COMMENT 'Product dimension with category hierarchy and unit cost.' AS
SELECT
  product_id,
  product_name,
  category,
  subcategory,
  unit_cost
FROM silver.products


Verify row count and distinct categories.

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT category) AS categories
FROM gold.dim_product


## Task 4 — fact_sales

Grain: one row per order line. Metrics: line_revenue = quantity × unit_price. line_margin = quantity × (unit_price − unit_cost). is_completed = true only for status = 'Completed'.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales
COMMENT 'Order-line fact table for governed sales measures.' AS
SELECT
  l.line_id, l.order_id, o.customer_id, l.product_id,
  o.order_date, o.status, o.channel,
  l.quantity, l.unit_price, p.unit_cost,
  CASE WHEN o.status = 'Completed' THEN true ELSE false END AS is_completed,
  l.quantity * l.unit_price                                  AS line_revenue,
  l.quantity * (l.unit_price - p.unit_cost)                 AS line_margin
FROM silver.order_lines l
JOIN  silver.sales_orders o ON l.order_id  = o.order_id
LEFT JOIN silver.products p ON l.product_id = p.product_id


Verify total rows, distinct orders, and completed revenue.

In [ ]:
%sql
SELECT
  COUNT(*)                                          AS total_rows,
  COUNT(DISTINCT order_id)                          AS distinct_orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue END), 2) AS completed_revenue
FROM gold.fact_sales


## Task 5 — fact_sales_dashboard (denormalized)

![Star schema vs flat table](../../assets/images/star_schema_vs_flat_table.png)

Denormalized flat table — all dimension attributes joined into one wide table. Power BI can work with just this one table for most visuals without needing to define relationships.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_dashboard
COMMENT 'Denormalized BI table joining fact_sales to conformed dimensions.' AS
SELECT
  f.line_id, f.order_id, f.order_date, f.status, f.channel,
  f.quantity, f.unit_price, f.unit_cost,
  f.is_completed, f.line_revenue, f.line_margin,
  d.year, d.quarter, d.month, d.year_month, d.day_of_week,
  c.customer_name, c.segment, c.customer_region,
  p.product_name, p.category, p.subcategory
FROM gold.fact_sales f
LEFT JOIN gold.dim_date     d ON f.order_date  = d.date_key
LEFT JOIN gold.dim_customer c ON f.customer_id = c.customer_id
LEFT JOIN gold.dim_product  p ON f.product_id  = p.product_id


Verify row count.

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM gold.fact_sales_dashboard


Preview the wide denormalized rows.

In [ ]:
%sql
SELECT * FROM gold.fact_sales_dashboard LIMIT 5


## Task 6 — Grain Check & Reconciliation

Verify that: (1) fact_sales_dashboard has no duplicate lines, (2) revenue in fact_sales = revenue in fact_sales_dashboard.

**Step 1 — grain check.** duplicates must be 0.

In [ ]:
%sql
SELECT
  COUNT(*)              AS total_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicates
FROM gold.fact_sales_dashboard


**Step 2 — revenue reconciliation.** Both columns must match.

In [ ]:
%sql
SELECT
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue END), 2) AS revenue_fact_sales_dashboard,
  (SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue END), 2) FROM gold.fact_sales)
    AS revenue_fact_sales
FROM gold.fact_sales_dashboard


**Step 3 — automated assert.** Raises an exception if either check fails.

In [ ]:
qry = ("SELECT ABS(SUM(CASE WHEN is_completed THEN line_revenue END) - "
       "(SELECT SUM(CASE WHEN is_completed THEN line_revenue END) FROM gold.fact_sales)) "
       "AS revenue_diff, COUNT(*) - COUNT(DISTINCT line_id) AS duplicates "
       "FROM gold.fact_sales_dashboard")
result = spark.sql(qry).collect()[0]
assert result["duplicates"] == 0, f"Grain check FAILED: {result['duplicates']} duplicate lines"
assert result["revenue_diff"] < 0.01, f"Revenue mismatch: {result['revenue_diff']}"


Print pass confirmation.

In [ ]:
print("[OK] Grain check passed — no duplicates")
print("[OK] Revenue reconciliation passed")


## Data Quality Reality Check — Silver vs Gold

Silver contains ALL order lines including Returned, Cancelled, and future-dated orders. Gold (fact_sales_dashboard) filters to Completed only. The difference is the business impact of data quality.

Compare Silver gross revenue against Gold completed revenue.

In [ ]:
%sql
SELECT 'Silver — all lines' AS source,
  COUNT(DISTINCT order_id) AS orders, ROUND(SUM(quantity * unit_price), 2) AS gross_revenue
FROM silver.order_lines UNION ALL
SELECT 'Gold — completed only' AS source,
  COUNT(DISTINCT order_id) AS orders, ROUND(SUM(line_revenue), 2) AS gross_revenue
FROM gold.fact_sales_dashboard WHERE is_completed = true


Status breakdown in Silver — shows how much revenue each order status contributes.

In [ ]:
%sql
SELECT status, COUNT(*) AS lines, ROUND(SUM(quantity * unit_price), 2) AS revenue
FROM silver.order_lines
GROUP BY status
ORDER BY revenue DESC


The gap between Silver gross revenue and Gold completed revenue = the revenue from non-Completed orders. This is why filtering matters: an unchecked Silver query would overstate revenue.

## Decision Log Prompt

Before leaving the workshop, write one row for `docs/templates/decision-log.md`:

| Decision | Options considered | Expected answer |
|---|---|---|
| Power BI should start from star schema or denormalized table? | star schema, flat dashboard table, direct Silver query | use `fact_sales_dashboard` for quick Day 2 demo, keep star schema as governed core |
| Where should completed-revenue logic live? | DAX, Power Query, Gold SQL | Gold SQL for shared definition; DAX only for presentation-specific variants |
| Is line-grain detail enough for all visuals? | detail only, aggregate later | detail is correct for drill-through; Day 2 adds KPI/aggregate objects |

This is not busywork. It prevents the same modeling debate from repeating in
each PBIX file.


## Self-Check — 5 Gold Objects

List all tables currently in the Gold schema.

In [ ]:
%sql
SHOW TABLES IN gold


Assert that all 5 required Gold objects exist. This cell must pass before Day 2.

In [ ]:
required = [f"{GOLD}.dim_date", f"{GOLD}.dim_customer", f"{GOLD}.dim_product",
            f"{GOLD}.fact_sales", f"{GOLD}.fact_sales_dashboard"]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    [print(" -", t) for t in missing]
    raise Exception("Workshop 1 incomplete. Fix missing tables before Day 2.")
print("[OK] All 5 Gold objects exist. Workshop 1 complete!")


Day 2 continuation note.

In [ ]:
print("     Tomorrow: Workshop 2 adds KPI layer → Power BI connector.")
